In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
GOLD_PATH = "abfss://gold@pravdatalake.dfs.core.windows.net"
GOLD_TABLE_PATH = f"{GOLD_PATH}/dimension_date"
GOLD_TABLE_NAME = "vehicle_sales.gold.dimension_date"

In [0]:
date_df = spark.sql("""
    SELECT explode(sequence(to_date('2000-01-01'), to_date('2026-12-31'), interval 1 day)) as full_date
""")

In [0]:
dimension_date = (
    date_df
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast(IntegerType()))
    .withColumn("year", year(col("full_date")))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("month", month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("day", dayofmonth(col("full_date")))
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))
    .withColumn("is_weekend", dayofweek(col("full_date")).isin([1, 7]))
)

In [0]:
dimension_date.display()

####Data Quality Checks

In [0]:
row_count = dimension_date.count()

In [0]:
duplicate_key_count = dimension_date.groupBy("date_key").count().filter("count > 1").count()

In [0]:
print(f"row count: {row_count}")
print(f"duplicate date_key count: {duplicate_key_count}")

In [0]:
assert duplicate_key_count == 0, "date_key should be unique in dim_date"

In [0]:
dimension_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_TABLE_PATH)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE_NAME}
    USING DELTA
    LOCATION '{GOLD_TABLE_PATH}'
""")

In [0]:
%sql
SELECT * FROM vehicle_sales.gold.dimension_date

In [0]:
spark.sql(f"OPTIMIZE {GOLD_TABLE_NAME}")